# The AI Telco Troubleshooting Challenge

Goal: Enhance the accuracy of Qwen3-32B when answering telco troubleshooting questions in telelogs data.

## I/ Set-up

### 1. Load libraries

In [1]:
import pandas as pd
import numpy as np

### 2.Load data

In [2]:
#load data
df = pd.read_csv("C:/Users/atanoh/Documents/Zindi telco challenge/train.csv")
print(df.head())

              ID                                           question answer
0  ID_1P7PJMPV0R  Analyze the 5G wireless network drive-test use...     C2
1  ID_8B1D1TUTFA  Analyze the 5G wireless network drive-test use...     C1
2  ID_IGGXMA9GZH  Analyze the 5G wireless network drive-test use...     C2
3  ID_D6C9N2X295  Analyze the 5G wireless network drive-test use...     C2
4  ID_8JC15PNP3Q  Analyze the 5G wireless network drive-test use...     C5


In [4]:
#Function to clean questions 
def clean_question(question):
    lines=question.split("\n")
    content=[]
    for line in lines:
        if "|" in line:
            content.append(line.split("|"))
    drive_test_data=[{"Observation": i+1,**dict(zip(content[0],row))} for i,row in enumerate(content[1:])]
    engineering_params=[{"Observation": i+1,**dict(zip(content[11],row))} for i,row in enumerate(content[12:])]
    #clean question
    ##recompute begining of the prompt
    q=""
    for l in [l+"\n" for l in lines[:19]]:
        q=q+l
    ##assemble
    cleaned_question= f" Question: {q} \nDrive test data: {drive_test_data} \nEngineering parameters {engineering_params} "
    return cleaned_question
#Apply to dataset
df["cleaned_question"]=df["question"].apply(lambda x: clean_question(x))
print(df["cleaned_question"][0])

 Question: Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \boxed{{}} in the final answer.

C1: The serving cell's downtilt angle is too large, causing weak coverage at the far end.
C2: The serving cell's coverage distance exceeds 1km, resulting in over-shooting.
C3: A neighboring cell provides higher throughput.
C4: Non-colocated co-frequency neighboring cells cause severe overlapping coverage.
C5: Frequent handovers degrade performance.
C6: Neighbor cell and serving cell have the same PCI mod 30, leading to interference.
C7: Test vehicle speed exceeds 40km/h, impacting user throughput.
C8: Average scheduled RBs are below 160, affecting throughput.

Given:
- The default electronic downtilt value is 255, representing a downtilt angle of 6 degrees. Other values repre

## II/ Untrained model

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

c:\Users\atanoh\Documents\Zindi telco challenge\telco\Telco-challenge\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:

# load the tokenizer and the model
model_name = "Qwen/Qwen3-32B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [ ]:
# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

In [ ]:
# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)